In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv(
    r"D:\project\MindSense\Early_Detection\data\raw\student_data.csv.csv"
)

# Remove duplicate header row
df = df[df["Timestamp"] != "Timestamp"]

# Convert timestamp
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Convert numeric columns
numeric_columns = [
    "Heart_Rate",
    "Blood_Pressure_Systolic",
    "Blood_Pressure_Diastolic",
    "Respiration_Rate",
    "Sleep_Duration",
    "Activity_Levels",
    "Cognitive_Load",
    "Study_Hours",
    "Academic_Stressors"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col])

# Sort by student and date
df = df.sort_values(["StudentID", "Timestamp"])

In [3]:
from sklearn.preprocessing import LabelEncoder

mood_encoder = LabelEncoder()
stress_encoder = LabelEncoder()
mental_encoder = LabelEncoder()

df["Mood"] = mood_encoder.fit_transform(df["Mood"])
df["Stress_Level"] = stress_encoder.fit_transform(df["Stress_Level"])
df["Mental_Health_Status"] = mental_encoder.fit_transform(df["Mental_Health_Status"])

In [4]:
features = [
    "Heart_Rate",
    "Blood_Pressure_Systolic",
    "Blood_Pressure_Diastolic",
    "Respiration_Rate",
    "Sleep_Duration",
    "Activity_Levels",
    "Mood",
    "Cognitive_Load",
    "Study_Hours",
    "Academic_Stressors",
    "Stress_Level"
]

target = "Mental_Health_Status"

In [5]:
scaler = MinMaxScaler()

df[features] = scaler.fit_transform(df[features])

In [6]:
df.head()

,StudentID,Profile,Timestamp,Heart_Rate,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Respiration_Rate,Sleep_Duration,Activity_Levels,Mood,Cognitive_Load,Study_Hours,Academic_Stressors,Stress_Level,Mental_Health_Status
0,1,High_Achiever,2015-01-01,0.664948,0.384937,0.345455,0.491525,0.777202,0.635534,0.666667,0.375,0.292683,0.000000,1.0,4
1,1,High_Achiever,2015-01-02,0.402062,0.523013,0.333333,0.372881,0.590674,0.454371,1.000000,0.250,0.401829,0.000000,1.0,2
2,1,High_Achiever,2015-01-03,0.386598,0.251046,0.412121,0.525424,0.792746,0.478707,0.333333,0.250,0.284756,0.111111,1.0,1
3,1,High_Achiever,2015-01-04,0.453608,0.288703,0.660606,0.457627,0.773748,0.413813,0.333333,0.375,0.377439,0.000000,1.0,1
4,1,High_Achiever,2015-01-05,0.422680,0.393305,0.424242,0.457627,0.753022,0.753493,0.666667,0.250,0.351220,0.222222,1.0,1


In [7]:
df.groupby("StudentID").size()

StudentID
1     3865
10    3865
2     3865
3     3865
4     3865
5     3865
6     3865
7     3865
8     3865
9     3865
dtype: int64

In [8]:
LOOKBACK = 60        # Previous 60 days
HORIZON = 90         # Predict 90 days later (3 months)

In [9]:
X = []
y = []

In [10]:
for student in df["StudentID"].unique():

    student_df = df[df["StudentID"] == student].reset_index(drop=True)

    feature_values = student_df[features].values
    target_values = student_df[target].values

    for i in range(len(student_df) - LOOKBACK - HORIZON + 1):

        X.append(feature_values[i:i+LOOKBACK])

        y.append(target_values[i+LOOKBACK+HORIZON-1])

In [11]:
import numpy as np

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(37160, 60, 11)
(37160,)


In [12]:
print(X.shape)
print(y.shape)

(37160, 60, 11)
(37160,)


In [13]:
import os
import numpy as np

In [14]:
np.save(
    r"D:\project\MindSense\Early_Detection\data\sequences\X.npy",
    X
)

np.save(
    r"D:\project\MindSense\Early_Detection\data\sequences\y.npy",
    y
)

In [15]:
import os

print(os.listdir(
    r"D:\project\MindSense\Early_Detection\data\sequences"
))

['X.npy', 'y.npy']


In [16]:
from tensorflow.keras.utils import to_categorical

y = to_categorical(y)

print(y.shape)

(37160, 6)


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    shuffle=False,
    random_state=42
)

In [18]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(23782, 60, 11)
(5946, 60, 11)
(7432, 60, 11)
(23782, 6)
(5946, 6)
(7432, 6)


In [19]:
import numpy as np

np.save(r"D:\project\MindSense\Early_Detection\data\processed\X_train.npy", X_train)
np.save(r"D:\project\MindSense\Early_Detection\data\processed\X_val.npy", X_val)
np.save(r"D:\project\MindSense\Early_Detection\data\processed\X_test.npy", X_test)

np.save(r"D:\project\MindSense\Early_Detection\data\processed\y_train.npy", y_train)
np.save(r"D:\project\MindSense\Early_Detection\data\processed\y_val.npy", y_val)
np.save(r"D:\project\MindSense\Early_Detection\data\processed\y_test.npy", y_test)

print("Datasets saved successfully!")

Datasets saved successfully!
